### debugging

In [7]:
import pandas as pd
from ast import literal_eval
import json

CSV_PATH = 'data/emplo.csv'
USE_COLS = [
    'employee_Id',       # your ID column
    'sector', 'contract_type', 'edu_value', 'city',
    'years_experience', 'salary', 'technical_skills'
]
HIERARCHY = ['sector', 'contract_type', 'edu_value', 'city', 'years_experience']

def preprocess():
    df = pd.read_csv(CSV_PATH, usecols=USE_COLS)
    df['sector'] = df['sector'].fillna('Unknown').astype(str)
    
    def split_skills(x):
        if pd.isna(x):
            return ['Unknown']
        try:
            lst = literal_eval(x)
            if isinstance(lst, (list, tuple)):
                return [str(s).strip() for s in lst]
        except Exception:
            pass
        return [s.strip() for s in str(x).split(',') if s.strip()]
    
    # explode skills into one-per-row
    df = df.assign(
        technical_skills=df['technical_skills'].apply(split_skills)
    ).explode('technical_skills')
    return df

def build_transition_model(df, level=0):
    if level >= len(HIERARCHY):
        return {}
    
    attr = HIERARCHY[level]
    nodes = {}
    
    for val, grp in df.groupby(attr, dropna=False):
        key = str(val) if not pd.isna(val) else 'Unknown'
        
        expected = {}
        # sector: list of all unique sectors in this cluster
        expected['sector'] = grp['sector'].fillna('Unknown').astype(str).unique().tolist()
        # contract_type: list of all unique contract types
        expected['contract_type'] = grp['contract_type'].fillna('Unknown').astype(str).unique().tolist()
        # edu_value: max in cluster
        expected['edu_value'] = float(grp['edu_value'].max())
        # city: list of all unique cities
        expected['city'] = grp['city'].fillna('Unknown').astype(str).unique().tolist()
        # years_experience: max in cluster
        expected['years_experience'] = float(grp['years_experience'].max())
        # salary: min in cluster
        expected['salary'] = float(grp['salary'].min())
        # technical_skills: union of all skills
        all_skills = set(grp['technical_skills'])
        expected['technical_skills'] = list(all_skills)
        
        extras = {}
        if level == len(HIERARCHY) - 1:
            unique_ids = sorted(set(grp['employee_Id']))
            extras['employee_ids'] = unique_ids
        
        node = {
            'expected_values': expected,
            **extras
        }
        # recurse
        node['clusters'] = build_transition_model(grp, level+1)
        nodes[key] = node
    
    return nodes

def print_model(model, depth=0, max_depth=2, indent=0):
    if depth > max_depth:
        return
    for k, v in model.items():
        print(' '*indent + f"Cluster: {k}")
        for a, val in v['expected_values'].items():
            if isinstance(val, list):
                items = ', '.join(str(x) for x in val)
                print(' '*(indent+2) + f"{a}: [{items}]")
            else:
                print(' '*(indent+2) + f"{a}: {val}")
        if 'employee_ids' in v:
            print(' '*(indent+2) + f"employee_ids: {v['employee_ids']}")
        print()
        print_model(v['clusters'], depth+1, max_depth, indent+4)

def export_transition_model(transition_model, file_path='transition_model.json'):
    """
    Exports the transition model as a JSON file.
    """
    with open(file_path, 'w') as json_file:
        json.dump(transition_model, json_file, indent=4)
    print(f"Transition model exported to {file_path}")

if __name__ == '__main__':
    df = preprocess()
    model = build_transition_model(df)
    print_model(model, max_depth=2)
    export_transition_model(model, file_path='transition_model.json')


Cluster: Aerospace & Defense
  sector: [Aerospace & Defense]
  contract_type: [Stage, Freelance, CDI, CDD, Alternance]
  edu_value: 20.0
  city: [El Tarf, Annaba, Ouargla, Blida, Béchar, Biskra, Tlemcen, Algiers, Tizi Ouzou, Chlef, Skikda, Laghouat, El Oued, Béjaïa, Illizi, Tamanrasset, Constantine, Saïda, M'sila, Aïn Témouchent, Tébessa, Oran, Jijel, Naama, Tiaret, Tindouf, Mostaganem, Ghardaïa, Djanet, In Salah, Batna, Sétif, Adrar, Timimoun]
  years_experience: 38.0
  salary: 35000.0
  technical_skills: [Problem Solving, Teamwork, Communication, Time Management]

    Cluster: Alternance
      sector: [Aerospace & Defense]
      contract_type: [Alternance]
      edu_value: 20.0
      city: [Skikda, Constantine, Blida, Tindouf, Saïda, Aïn Témouchent, Mostaganem, Béchar, Jijel, El Oued, Ouargla, Illizi, Biskra, Tiaret, Laghouat, M'sila, Adrar]
      years_experience: 38.0
      salary: 44100.0
      technical_skills: [Problem Solving, Teamwork, Time Management, Communication]

        

In [ ]:
import json

# Export the transition model to a JSON file
def export_transition_model(transition_model, file_path='transition_model.json'):
    """
    Exports the transition model as a JSON file.

    Args:
        transition_model (dict): The transition model to export.
        file_path (str): The file path to save the JSON file.
    """
    with open(file_path, 'w') as json_file:
        json.dump(transition_model, json_file, indent=4)
    print(f"Transition model exported to {file_path}")

# Import the transition model from a JSON file


# Example usage:
# Export the transition model
export_transition_model(transition_model, file_path='transition_model.json')

# Import the transition model in another notebook
# (Run this in the other notebook)
# imported_model = import_transition_model(file_path='transition_model.json')

In [ ]:

# use this function in the other files to import the json file

def import_transition_model(file_path='transition_model.json'):
    """
    Imports the transition model from a JSON file.

    Args:
        file_path (str): The file path of the JSON file to import.

    Returns:
        dict: The imported transition model.
    """
    with open(file_path, 'r') as json_file:
        transition_model = json.load(json_file)
    print(f"Transition model imported from {file_path}")
    return transition_model

In [30]:
import pandas as pd
import json
from ast import literal_eval

CSV_PATH = 'data/jobs.csv'
USE_COLS = [
    'job_Id',
    'sector',
    'type of contract',
    'edu_value',
    'job location',
    'experience_min_req',
    'experience_max_req',
    'salary',
    'technical_skills'
]
HIERARCHY = ['sector', 'contract_type', 'edu_value', 'city', 'years_experience']


def preprocess_jobs():
    df = pd.read_csv(CSV_PATH, usecols=USE_COLS)
    df = df.rename(columns={
        'type of contract': 'contract_type',
        'job location': 'city',
        'experience_min_req': 'exp_min',
        'experience_max_req': 'exp_max',
    })
    df['years_experience'] = ((df['exp_min'].fillna(0) + df['exp_max'].fillna(0)) / 2).round(1)
    df['sector'] = df['sector'].fillna('Unknown').astype(str)
    df['city'] = df['city'].fillna('Unknown').astype(str)

    def split_skills(x):
        if pd.isna(x):
            return ['Unknown']
        try:
            lst = literal_eval(x)
            if isinstance(lst, (list, tuple)):
                return [s.strip() for s in lst]
        except Exception:
            pass
        return [s.strip() for s in str(x).split(',') if s.strip()]

    df = df.assign(
        technical_skills=df['technical_skills'].apply(split_skills)
    ).explode('technical_skills')
    return df[['job_Id', 'sector', 'contract_type', 'edu_value', 'city', 'years_experience', 'salary', 'technical_skills']]


def build_job_transition_model(df, level=0):
    if level >= len(HIERARCHY):
        return {}
    attr = HIERARCHY[level]
    nodes = {}
    for val, grp in df.groupby(attr, dropna=False):
        key = str(val) if pd.notna(val) else 'Unknown'
        expected = {}
        # sector: list of all unique sectors
        expected['sector'] = grp['sector'].fillna('Unknown').astype(str).unique().tolist()
        # contract_type: list of all unique contract types
        expected['contract_type'] = grp['contract_type'].fillna('Unknown').astype(str).unique().tolist()
        # edu_value: max
        expected['edu_value'] = float(grp['edu_value'].max())
        # city: list
        expected['city'] = grp['city'].fillna('Unknown').astype(str).unique().tolist()
        # years_experience: max
        expected['years_experience'] = float(grp['years_experience'].max())
        # salary: max
        expected['salary'] = float(grp['salary'].max())
        # technical_skills: union
        all_skills = set(grp['technical_skills'])
        expected['technical_skills'] = list(all_skills)

        extras = {}
        if level == len(HIERARCHY) - 1:
            extras['job_ids'] = sorted(grp['job_Id'].unique().tolist())

        node = {'expected_values': expected, **extras}
        node['clusters'] = build_job_transition_model(grp, level+1)
        nodes[key] = node
    return nodes


def print_model(model, depth=0, max_depth=2, indent=0):
    if depth > max_depth:
        return
    for k, v in model.items():
        print(' '*indent + f"Cluster: {k}")
        for a, val in v['expected_values'].items():
            if isinstance(val, list):
                items = ', '.join(str(x) for x in val)
                print(' '*(indent+2) + f"{a}: [{items}]")
            else:
                print(' '*(indent+2) + f"{a}: {val}")
        if 'job_ids' in v:
            print(' '*(indent+2) + f"job_ids: {v['job_ids']}")
        print()
        print_model(v['clusters'], depth+1, max_depth, indent+4)


def export_transition_model(model, file_path='job_transition_model.json'):
    with open(file_path, 'w') as f:
        json.dump(model, f, indent=2)
    print(f"Exported job transition model to {file_path}")


if __name__ == '__main__':
    df_jobs = preprocess_jobs()
    job_model = build_job_transition_model(df_jobs)
    print_model(job_model, max_depth=2)
    export_transition_model(job_model)


Cluster: Aerospace & Defense
  sector: [Aerospace & Defense]
  contract_type: [Internship, CDI, Alternance, Freelance, CDD]
  edu_value: 20.0
  city: [Sétif, Blida, Laghouat, Algiers, Annaba, Oran, Batna, Constantine, Biskra, Jijel, Khenchela, Mostaganem, Bouira, El Oued, Tizi Ouzou, Ghardaïa]
  years_experience: 11.5
  salary: 180000.0
  technical_skills: [Flight Testing, Jet Propulsion, Avionics Systems, Ground Control Systems, Defense Contracting, Satellite Communications, Systems Engineering, Mission Planning, CAD for Aerospace, Aerospace Materials, Reliability Engineering, DO-178C Certification, Flight Control Systems, Thermal Analysis, Radar Systems, Aerodynamics Simulation, Unmanned Aerial Systems, Military Standards (MIL-STD), ITAR Compliance, Weapon Systems Integration]

    Cluster: Alternance
      sector: [Aerospace & Defense]
      contract_type: [Alternance]
      edu_value: 20.0
      city: [Laghouat, Annaba, Batna, Constantine, Oran, Blida, Bouira, Algiers]
      years_